In [5]:
import json,time, os

import requests, feedparser, xmltodict
import polars as pl
from config_info import APIS
from pprint import pprint

from scrapers.arxiv import parser_arxiv
from scrapers.hal import parser_hal

# Basic Fetch

In [6]:
def fetch_raw(url, headers=None):
    headers = headers or {"User-Agent":  "IntelliCorpus/1.0 (contact: paull@scholar-cergy.com)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    content_type = r.headers.get("Content-Type","")
    if 'xml' in content_type:
        return r.text
    elif 'json' in content_type:
        return r.json()
    else:
        return r.text

# Semantic Scholar

In [7]:
def semantic_fetch(results):
    for result in results["data"]:
        print(result)
        # query = APIS["Semantic Scholar"]["paper_url"].format(paper_id=result["paperId"])
        # res = fetch_raw(query)
        # print(res)
        # print(query)

# CORE

In [4]:
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.getenv("CORE_API_KEY")
# url = "https://api.core.ac.uk/v3/search/works"

# headers = {
#     "Authorization": f"Bearer {API_KEY}"
# }

# params = { 
#     "q": "AI agent",
#     "limit": 5
# }

# r = requests.get(url, headers=headers, params=params)
# r.raise_for_status()

# data = r.json()
# # print(data.keys())
# pprint(data)


# Main 

In [8]:
# raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
# result_arxiv =  parser_arxiv(raw_result)
# print(type(result_arxiv)) # Arxiv Ok
 
# Test HAL
# url_hal = fetch_raw(APIS["HAL"]["api_url"].format(query="AI agent",quantity='2'))
# result_hal = parser_hal(url_hal)
# pprint(result_hal[0]) #Good 

# Test PubMed
from scrapers.pubmed import fetch_pubmed
xml_batches_pubmed = fetch_pubmed(
    query="AI agent",
    max_results=1,
    batch_size=1,
    email="proliquer@scholar-perigueuxu.com"
)
pprint(xml_batches_pubmed)

# Test Sementic Scholar
# url_sem = fetch_raw(APIS["Semantic Scholar"]["api_url"].format(query="AI agent",quantity='5'))
# print(type(url_sem))
# semantic_fetch(url_sem)


['<?xml version="1.0" ?>\n'
 '<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January '
 '2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n'
 '<PubmedArticleSet>\n'
 '<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" '
 'IndexingMethod="Automated"><PMID '
 'Version="1">41999042</PMID><DateCompleted><Year>2026</Year><Month>04</Month><Day>18</Day></DateCompleted><DateRevised><Year>2026</Year><Month>04</Month><Day>18</Day></DateRevised><Article '
 'PubModel="Print"><Journal><ISSN '
 'IssnType="Electronic">2051-817X</ISSN><JournalIssue '
 'CitedMedium="Internet"><Volume>14</Volume><Issue>8</Issue><PubDate><Year>2026</Year><Month>Apr</Month></PubDate></JournalIssue><Title>Physiological '
 'reports</Title><ISOAbbreviation>Physiol '
 'Rep</ISOAbbreviation></Journal><ArticleTitle>Pulse waveform analysis in '
 'Zambian adults living with HIV on antiretroviral therapy: A cross-sectional '
 'study of vascular '
 'dysfunction.</ArticleTitle><Pagi

In [11]:
from scrapers.pubmed import format_pubmed_data
pubmed_list = format_pubmed_data(xml_batches_pubmed)


In [ ]:
# from models.postgres.corpus_schema import metadata, document_table
# from config.db_engine import get_db_engine
# metadata.create_all(get_db_engine())

In [25]:
from database.postgres.crud import upsert_data
from models.postgres.corpus_schema import document_table 
from config.db_engine import get_db_engine
from processing.cleaning_data import normalize_data

arxiv_mapping = {
    "published_at": "published",
}
raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
result_arxiv =  parser_arxiv(raw_result)
clean_arxiv_data = normalize_data(
    raw_data=result_arxiv, 
    source_name="arXiv",
    date_columns=["published_at"],
    columns_drop=["updated"]
    )


# hal_mapping = { "published" : "published_at"}
# clean_hal_data = normalize_data(
#     raw_data=result_hal,
#     source_name="Hal",
#     column_mapping=hal_mapping,
#     date_columns=["published"]
# )

clean_pubmed_data = normalize_data(
    raw_data= pubmed_list,
    source_name="Pubmed",
    date_columns=["published"]
)
pprint(clean_pubmed_data)
engine = get_db_engine()
# upsert_data(clean_pubmed_data, ['id'], document_table, engine)
# upsert_data(clean_pubmed_data, ['id'], document_table, engine)


[{'authors': ['Theresa Chikopela',
              'Longa Kaluba',
              'Shirley Mwaanga',
              'Fastone M Goma'],
  'id': '1b81b305-9375-5177-a6a8-657a95bf0759',
  'pdf_url': None,
  'published_at': {'Day': '18', 'Month': '04', 'Year': '2026'},
  'source': 'PubMed',
  'summary': {'#text': 'People living with HIV (PLWH) experience a greater '
                       'risk of cardiovascular disease due to HIV-related '
                       'vascular injury. Pulse waveform analysis can '
                       'characterize arterial vascular dysfunction and this '
                       'study compared arterial waveforms in PLWH on '
                       'antiretroviral therapy (ART) with HIV-negative '
                       'controls. In this cross-sectional study, participants '
                       'were recruited from the University Teaching Hospitals, '
                       'Lusaka (September 2018-June 2019). 55 PLWH on ART '
                       '≥2\u2009y